# 00b — Chạy ABLATION và SEED trên Colab

Dùng notebook này khi máy cá nhân **quá chậm hoặc cần dùng việc khác**.
Hai phần nặng nhất của dự án được đẩy lên đây:

| Phần | Trên MacBook | Trên Colab T4 |
|---|---|---|
| 30 run ablation (ngân sách 15 epoch) | ~5 giờ | **~1,5–2 giờ** |
| 2 seed cho M2 | ~1,2 giờ | **~25 phút** |

Phần nhẹ (nén int8, thí nghiệm rò rỉ, tổng hợp báo cáo) vẫn chạy ở máy — chúng nhanh
và cần đo latency trên **máy đích**, không đo trên Colab được.

> ⚠️ `Runtime → Change runtime type → T4 GPU → Save` trước khi chạy.

## Ô 1 — Kiểm GPU

In [ ]:
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit(
        "CHƯA BẬT GPU. Runtime -> Change runtime type -> T4 GPU -> Save, rồi chạy lại.\n"
        "Chạy ablation trên CPU mất hàng chục tiếng."
    )
print("GPU:", torch.cuda.get_device_name(0),
      f"| VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

## Ô 2 — Google Drive (để kết quả sống sót khi mất session)

Ablation chạy 1,5–2 giờ. Colab miễn phí hay ngắt giữa chừng. `artifacts/` trỏ sang
Drive nên mất session chỉ cần chạy lại — `run_ablation.py` tự bỏ qua run đã xong.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/gtsrb")
(DRIVE / "artifacts" / "runs").mkdir(parents=True, exist_ok=True)
print("Thư mục Drive:", DRIVE)
done = sorted((DRIVE / "artifacts" / "runs").glob("*"))
print(f"Đã có {len(done)} run trên Drive")

## Ô 3 — Đưa code lên

Trên máy bạn chạy `make zip` rồi upload `gtsrb_code.zip` ở ô này.

In [ ]:
import os, zipfile, shutil
from pathlib import Path
PROJECT = Path("/content/DLprj")

if PROJECT.exists():
    print("Đã có code. Xoá bằng  !rm -rf /content/DLprj  nếu muốn upload lại.")
else:
    from google.colab import files
    print("Chọn gtsrb_code.zip:")
    up = files.upload()
    name = next(iter(up))
    PROJECT.mkdir(parents=True)
    with zipfile.ZipFile(name) as z: z.extractall(PROJECT)
    os.remove(name)

os.chdir(PROJECT)

# artifacts/ -> Drive
local = PROJECT / "artifacts"
if not local.is_symlink():
    if local.exists(): shutil.rmtree(local)
    local.symlink_to(DRIVE / "artifacts")
print("artifacts/ ->", (DRIVE / "artifacts"))
print("thư mục làm việc:", os.getcwd())

## Ô 4 — Cài gói còn thiếu

★ **Không `pip install torch`** — sẽ kéo về bản không khớp CUDA và bạn chạy trên CPU
mà không biết.

In [ ]:
!pip install -q -r requirements-colab.txt
import thop; print("thop", thop.__version__)

## Ô 5 — Dữ liệu (~4 phút)

Tải thẳng trên Colab chứ không upload. Để ở `/content` chứ **không** để trên Drive —
đọc từ Drive lúc train chậm hơn nhiều.

In [ ]:
!python scripts/prepare_data.py --download --preprocess clahe --img-size 48

In [ ]:
# Kiểm hai con số bắt buộc trước khi train bất cứ thứ gì
import sys; sys.path.insert(0, "src")
import pandas as pd
from gtsrb.data.split import check_leakage
f = pd.read_csv("data/processed/index.csv")
r = check_leakage(f)
print(f"tổng ảnh: {len(f)}  (phải 51839)")
print(f"track dùng chung: {r['n_shared_tracks']}  (PHẢI 0)")
assert len(f) == 51839 and r["n_shared_tracks"] == 0, "dữ liệu không sạch, dừng lại"
print("OK")

## Ô 6 — ABLATION 30 run

Trục `resolution` và `preprocess` cần dựng thêm cache (mỗi cache ~11 giây) —
`run_ablation.py` tự lo.

★ Ô này an toàn khi chạy lại: run đã có `result.json` sẽ bị bỏ qua.

In [ ]:
!python scripts/run_ablation.py --axes all --budget \
    --set train.num_workers=2

## Ô 7 — Thêm 2 seed cho M2

Để trả lời "chênh lệch giữa các model có vượt nhiễu khởi tạo không".

In [ ]:
!python scripts/run_seeds.py --config configs/m2_vggres.yaml --seeds 43 44 \
    --set train.num_workers=2

## Ô 8 — Đánh giá các run mới trên tập test

Bắt buộc: không có số test thì bảng ablation trống.

In [ ]:
!python scripts/evaluate.py --runs "artifacts/runs/*" --no-mcnemar

## Ô 9 — Đóng gói mang về

★ Chỉ lấy `result.json`, `config.yaml`, `train_log.csv` — **không lấy checkpoint**.
Ablation chỉ cần số liệu, mà 30 checkpoint là hàng GB. Gói này chỉ vài trăm KB.

In [ ]:
import zipfile
from pathlib import Path

KEEP = ("result.json", "config.yaml", "train_log.csv")
out = Path("/content/ablation_seeds_results.zip")
runs = [r for r in sorted(Path("artifacts/runs").glob("*"))
        if (r / "result.json").exists()]

with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    n = 0
    for run in runs:
        for name in KEEP:
            p = run / name
            if p.exists():
                z.write(p, f"artifacts/runs/{run.name}/{name}")
        n += 1

print(f"Đóng gói {n} run -> {out.name} ({out.stat().st_size/1024:.0f} KB)")
from google.colab import files
files.download(str(out))

---

## Mang kết quả về máy

```bash
cd ~/Downloads/DLprj
python scripts/merge_runs.py ~/Downloads/ablation_seeds_results.zip
```

`merge_runs.py` chỉ chép run chưa có, kiểm `result.json` hợp lệ trước khi chép, và
báo rõ cái nào bỏ qua vì sao.

Rồi tổng hợp lại:

```bash
python scripts/run_ablation.py --collect-only   # bảng + 6 hình ablation
python scripts/run_seeds.py --collect-only      # mean±std, và chênh lệch có vượt nhiễu không
make report baocao
```

## Nếu mất session giữa chừng

Chạy lại Ô 1 → 5, rồi Ô 6. `run_ablation.py` tự bỏ qua run đã có `result.json`
trên Drive, nên nó làm tiếp từ chỗ dở.

## Vì sao không đo tốc độ trên Colab

Latency của T4 không nói gì về thiết bị biên. `make speed` và `make edge` phải chạy
trên **máy đích**, và lúc đo phải tắt mọi thứ khác.